In [1]:
import time

import pandas

startup_start_time = time.time()

import torch
from transformers import DistilBertForSequenceClassification, DistilBertTokenizer
from captum.attr import LayerIntegratedGradients

tokenizer_distil = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

model_distil = DistilBertForSequenceClassification.from_pretrained("./final_distilbert-model/")
model_distil.eval()

# Detect MPS (Apple Silicon GPU)
force_cpu = False
device = torch.device("mps" if torch.backends.mps.is_available() and not force_cpu else "cpu")
print(f"Using device: {device}")
model_distil.to(device)
startup_end_time = time.time()
print(f"Startup time: {startup_end_time - startup_start_time}")

Using device: cpu
Startup time: 10.454726457595825


In [2]:
def forward(_input_ids, _attention_mask, _model,):
    out = _model(input_ids=_input_ids, attention_mask=_attention_mask).logits
    return out

In [3]:
lig_distilbert = LayerIntegratedGradients(forward, model_distil.distilbert.embeddings)

In [4]:
def compute_attributions(_model, _tokz, _lig, _text):
    _inputs = _tokz(_text, return_tensors="pt", truncation=True, padding='max_length', max_length=512)
    _inputs.to(device)
    _input_ids = _inputs['input_ids']
    _input_ids.to(device)
    _attention_mask = _inputs['attention_mask']
    _attention_mask.to(device)
    baseline = torch.zeros_like(_input_ids)
    baseline.to(device)
    
    with torch.no_grad():
        _logits = _model(**_inputs).logits
        target = torch.argmax(_logits, dim=1).item()
        conf = torch.softmax(_logits, dim=1)[0][target].item()
    
    _attributions, _delta = _lig.attribute(
        inputs=_input_ids,
        baselines=baseline,
        additional_forward_args=(_attention_mask,_model),
        target=target,
        return_convergence_delta=True
    )
    return _input_ids, target, conf, _attributions, _delta

In [5]:
def merge_wordpieces(tokens, scores):
    merged_tokens = []
    merged_scores = []
    current_token, current_score = "", 0.0

    for token, score in zip(tokens, scores):
        if token.startswith("##"):
            current_token += token[2:]
            current_score += abs(score)
        else:
            if current_token:
                merged_tokens.append(current_token)
                merged_scores.append(current_score)
            current_token = token
            current_score = abs(score)
    if current_token:
        merged_tokens.append(current_token)
        merged_scores.append(current_score)
    return merged_tokens, merged_scores

In [6]:
def show_topk_important_words(_attributions, _input_ids, k=5, verbose=False):
    tokens = tokenizer_distil.convert_ids_to_tokens(_input_ids[0])
    scores = torch.linalg.norm(_attributions, dim=-1,).squeeze(0).tolist()
    filtered = [(t, s, tid) for t, s, tid in zip(tokens, scores, _input_ids[0].tolist()) if t not in ["[CLS]", "[SEP]", "[PAD]", '.', ',', "(", ")"]]

    top_k = sorted(filtered, key=lambda x: abs(x[1]), reverse=True)[:k]
    return top_k

In [7]:
line = "---------------------------------------------------------------------------------------------------------------"
def compare_importances(_text, k = 15, verbose = True):
    input_ids, pred_label, pred_prob,attributions, delta, = compute_attributions(model_distil, tokenizer_distil, lig_distilbert, _text)
    words, _, top_k_ids,  = zip(*show_topk_important_words(attributions, input_ids, k = k))
    if verbose:
        print("Predicted label {} | Confidence: {} |Top {} important words:".format(pred_label, pred_prob, k))
    print("TOP-K: "," | ".join(words),"\n")
    if verbose:
        print(line)
    return input_ids, top_k_ids, pred_label, pred_prob, words

In [8]:
from read_jsonl import read_jsonl
eval_df = read_jsonl("DB-bio/combined_val_and_val_sft_anonymized.jsonl")
len(eval_df)

486

In [9]:
import time
runtimes = []

eval_results = [[], []]

sample = eval_df.sample(frac=0.005, random_state=69)
print(line)
top_k_words_results = []
for i, row in sample.iterrows():
    print("Sample [{}]:".format(i))
    text = row["text"]
    print(text)
    print("True Label: {}\n".format(row["label"]))

    start_time = time.time()
    ids, top_k_ids, target, conf, top_k_words = compare_importances(text, verbose=False)
    top_k_words_results.append((i, text, top_k_words))
    end_time = time.time()
    runtimes.append(end_time - start_time)
    print("Model original prediction: \n", target, conf,"\n")
    # evaluate results
    for tid in top_k_ids:
        ids[ids == tid] = 103
    _logits = model_distil(ids, attention_mask=torch.ones_like(ids).to(device)).logits
    target_eval = torch.argmax(_logits, dim=1).item()
    conf_eval = torch.softmax(_logits, dim=1)[0][target_eval].item()
    print("Model w/out top-k: \n",target_eval, conf_eval,"\n")
    print(tokenizer_distil.decode(ids[0].tolist()))
    print(line)
    eval_results[target].append((target_eval, conf - conf_eval))
    
print(sum(runtimes) / len(runtimes))
print(eval_results)

---------------------------------------------------------------------------------------------------------------
Sample [436]:
A person (Date of Birth – Date of Death) was a U.S. Representative from a state in the Midwest. Born in a small town in the Midwest, this person attended the public schools and the local high school. They graduated from a state university in the Midwest in Year and from the law department of the same university in Year. They were admitted to the bar in Year and commenced practice in their hometown. This person served as district attorney of the county from Year–Year. They served as delegate to the political party State conventions in Years. They served as member of the board of regents of the state university in Years. This person served as member of the state Senate Year–Year. Elected as a member of a political party to the national legislature (Term Start Date – Term End Date), they represented a congressional district. On a specific date, they were one of the

In [10]:
pandas.DataFrame(top_k_words_results).to_csv("DB-bio/top-k-words-gradient")

In [24]:
test_ids = torch.zeros_like(ids).to(device)
test_ids += 103
test_ids[0][1:10] = 2711
attention_mask = torch.ones_like(ids).to(device)
_logits = model_distil(test_ids, attention_mask=attention_mask).logits
target_eval = torch.argmax(_logits, dim=1).item()
conf_eval = torch.softmax(_logits, dim=1)[0][target].item()

In [25]:
target_eval, conf_eval

(1, 0.5817022919654846)

In [21]:
tokenizer_distil.decode([i for i in range(0, 1000)])

'[PAD] [unused0] [unused1] [unused2] [unused3] [unused4] [unused5] [unused6] [unused7] [unused8] [unused9] [unused10] [unused11] [unused12] [unused13] [unused14] [unused15] [unused16] [unused17] [unused18] [unused19] [unused20] [unused21] [unused22] [unused23] [unused24] [unused25] [unused26] [unused27] [unused28] [unused29] [unused30] [unused31] [unused32] [unused33] [unused34] [unused35] [unused36] [unused37] [unused38] [unused39] [unused40] [unused41] [unused42] [unused43] [unused44] [unused45] [unused46] [unused47] [unused48] [unused49] [unused50] [unused51] [unused52] [unused53] [unused54] [unused55] [unused56] [unused57] [unused58] [unused59] [unused60] [unused61] [unused62] [unused63] [unused64] [unused65] [unused66] [unused67] [unused68] [unused69] [unused70] [unused71] [unused72] [unused73] [unused74] [unused75] [unused76] [unused77] [unused78] [unused79] [unused80] [unused81] [unused82] [unused83] [unused84] [unused85] [unused86] [unused87] [unused88] [unused89] [unused90] [u

tensor([[0, 0, 0,  ..., 0, 0, 0]], dtype=torch.int32)

In [65]:
token_ids = torch.zeros((1,513), dtype=torch.int)
attention_mask = torch.ones_like(token_ids, dtype=torch.float)
with torch.no_grad():
    print(model_distil(input_ids=token_ids, attention_mask=attention_mask).logits)

RuntimeError: The size of tensor a (513) must match the size of tensor b (512) at non-singleton dimension 1

RuntimeError: The size of tensor a (1600) must match the size of tensor b (512) at non-singleton dimension 1